[![cloudevel](../img/cloudevel.png)](https://cloudevel.com)

# Solución: imágenes de contenedor y registro

**Cómo leer este cuaderno.** Cada ejercicio reproduce el enunciado del cuaderno original y presenta su solución.

* Las celdas **sin etiqueta** se ejecutan en cualquier equipo, sin credenciales ni cuenta de *GitHub*. Escriben solo en carpetas temporales.
* Las celdas con la etiqueta `skip-execution` necesitan una cuenta de *GitHub*, un proyecto de nube o un servidor; la integración continua (*CI*) no las ejecuta. Debajo de cada una se describe la salida esperada.
* Ningún valor real aparece en el código: los datos de cada persona se leen de variables de entorno o se sustituyen por los marcadores indicados.

Las celdas con etiqueta `requiere-docker` necesitan un demonio de *Docker* (el *Codespace* lo trae con *Docker-in-Docker*). Para que la solución se pueda comprobar sin *Docker*, el ejercicio 1 incluye un ensayo equivalente que arranca la *API* con el mismo servidor y las mismas opciones que la imagen.

La siguiente celda importará el módulo `utilidades` (que está en esta misma carpeta) y localizará la raíz del repositorio. Todas las comprobaciones locales de este cuaderno leen archivos desde esa raíz.

In [ ]:
import utilidades as u

RAIZ = u.raiz_repositorio()
RAIZ.name

## Ejercicio 1. Construir la imagen y comprobar GET /health

Construir la imagen en el *Codespace* y comprobar que `GET /health` responde con `"almacen": "memoria"` y `"status": "ok"`.

La siguiente celda construirá la imagen `cd441-api` desde la raíz del repositorio, la ejecutará en segundo plano publicando el puerto 8080 con `APP_VERSION=local`, consultará la ruta de salud y detendrá el contenedor. El `--rm` lo elimina al detenerse.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
docker build -t cd441-api .
docker run -d --rm --name cd441-api -p 8080:8080 -e APP_VERSION=local cd441-api
sleep 3
docker ps --filter name=cd441-api
curl -fsS http://127.0.0.1:8080/health
echo
docker stop cd441-api

**Salida esperada.** `docker ps` muestra el contenedor `cd441-api` con estado `Up`. La respuesta de `curl` es `{"almacen":"memoria","status":"ok","version":"local"}`. Un segundo `docker build` sin cambios muestra las capas con `CACHED`.

**Ensayo equivalente sin Docker.** La instrucción `CMD` del `Dockerfile` ejecuta `gunicorn --bind 0.0.0.0:${PORT} --workers 1 --threads 4 --timeout 30 wsgi:app`. Las siguientes celdas hacen lo mismo con un entorno virtual temporal y un puerto libre.

La siguiente celda creará una copia del repositorio y un entorno virtual con las dependencias de ejecución y desarrollo (requiere acceso a *PyPI*).

In [ ]:
COPIA = u.copiar_repositorio(RAIZ, u.directorio_temporal())
PYTHON = u.crear_entorno_api(COPIA / "api", COPIA.parent)
print("Entorno listo")

La siguiente celda arrancará *gunicorn* con las mismas opciones del `Dockerfile` en un puerto libre, esperará a que responda, consultará `GET /health` con `urllib` y lo detendrá. Comprobará los tres campos de la respuesta.

In [ ]:
import json
import os
import socket
import subprocess
import time
import urllib.request

with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    PUERTO = s.getsockname()[1]

servidor = subprocess.Popen(
    [str(PYTHON), "-m", "gunicorn", "--bind", f"127.0.0.1:{PUERTO}", "--workers", "1",
     "--threads", "4", "--timeout", "30", "wsgi:app"],
    cwd=COPIA / "api",
    env={"PATH": os.environ["PATH"], "PORT": str(PUERTO), "APP_VERSION": "local"},
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
try:
    cuerpo = None
    for _ in range(30):
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{PUERTO}/health", timeout=2) as respuesta:
                estado, cuerpo = respuesta.status, json.loads(respuesta.read())
            break
        except OSError:
            time.sleep(0.5)
finally:
    servidor.terminate()
    servidor.wait(timeout=10)

print(estado, cuerpo)
assert estado == 200
assert cuerpo == {"almacen": "memoria", "status": "ok", "version": "local"}

## Ejercicio 2. Revisar la construcción y la publicación en una ejecución de CI

Revisar en una ejecución del *workflow* `CI` el paso `Construir imagen (sin publicar)` del *job* `verificar` y, en un *push* a `main`, el *job* `publicar-imagen`: la etiqueta debe ser el *SHA* del *commit*, no `latest`.

La siguiente celda comprobará en `ci.yml` que el paso de construcción de `verificar` ejecuta `docker build` sin `docker push`, y que `publicar-imagen` publica con `push: true` y la etiqueta formada con `github.sha`, sin la palabra `latest`.

In [ ]:
wf = u.cargar_workflow(RAIZ / ".github" / "workflows" / "ci.yml")

construir = next(p for p in u.pasos(wf, "verificar") if p.get("name") == "Construir imagen (sin publicar)")
print("verificar ->", construir["run"])
assert "docker build" in construir["run"] and "push" not in construir["run"]

publicar = next(p for p in u.pasos(wf, "publicar-imagen") if p.get("uses", "").startswith("docker/build-push-action@"))
etiquetas = publicar["with"]["tags"]
print("publicar  ->", etiquetas, "| push:", publicar["with"]["push"])
assert publicar["with"]["push"] is True
assert "github.sha" in etiquetas and "latest" not in etiquetas

La siguiente celda consultará en *GHCR* las etiquetas de la última versión publicada del paquete `api`. Requiere que la sesión de `gh` tenga el permiso `read:packages` (`gh auth refresh --scopes read:packages`).

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
REPO="$(gh repo view --json nameWithOwner --jq .nameWithOwner)"
PROPIETARIO="${REPO%%/*}"
NOMBRE="${REPO#*/}"
gh api "users/${PROPIETARIO}/packages/container/${NOMBRE}%2Fapi/versions" \
  --jq '.[0].metadata.container.tags'
git rev-parse HEAD

**Salida esperada.** Una lista con una sola etiqueta de 40 caracteres hexadecimales, igual al *SHA* del *commit* que disparó la publicación (el que imprime `git rev-parse HEAD` si no hay *commits* posteriores), y sin `latest`. Si el propietario es una organización, se sustituye `users/` por `orgs/`.

## Ejercicio 3. Reconocer instrucciones del Dockerfile

Abrir el `Dockerfile` y señalar qué instrucción impide que el proceso corra como *root*, cuál fija el *digest* de la imagen base y cuál declara el `HEALTHCHECK`.

La siguiente celda buscará cada instrucción con una expresión regular, mostrará su número de línea y comprobará las tres respuestas: `USER` con un identificador numérico, `FROM` con `@sha256:` y 64 caracteres hexadecimales, y `HEALTHCHECK` con su orden `CMD`.

In [ ]:
import re

lineas = (RAIZ / "Dockerfile").read_text(encoding="utf-8").splitlines()
busquedas = {
    "usuario sin privilegios": r"^USER\s+\d+:\d+\s*$",
    "imagen base con digest": r"^FROM\s+\S+@sha256:[0-9a-f]{64}\s*$",
    "comprobación de salud": r"^HEALTHCHECK\b",
}
for respuesta, patron in busquedas.items():
    numeros = [i for i, linea in enumerate(lineas, start=1) if re.match(patron, linea)]
    assert len(numeros) == 1, (respuesta, numeros)
    print(f"{respuesta:26} línea {numeros[0]:3}: {lineas[numeros[0] - 1][:70]}")

El número `10001` es un identificador distinto de 0, que es el de *root*. Por eso la instrucción `USER 10001:10001` impide el acceso como administrador. Un identificador numérico permite que plataformas como *Kubernetes* lo verifiquen sin leer `/etc/passwd`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023-2026.</p>